# 1.Data Cleaning 
 ---

**Assignment:** *Machine Learning 2026 — NOVA School of Business and Economics*  
**Dataset:** Predict students' dropout and academic success   
**Goal:** Identify high-risk students at the end of their first semester using: Socio-economic background, Enrollment data and 1st semester academic performance


 
 *71907 Lavinia Antonino | 71993 Mª Teresa Silva | 73171 Leonardo Cantu | 72731 Edoardo Sirianni | 71947 Carolina Diogo*

 ---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Data Understanding & Preparation](#2-data-understanding--preparation)
   - [2.1 Load the Dataset & Preview](#21-load-the-dataset--preview)
   - [2.2 Inspect Data Structure](#22-inspect-data-structure)
   - [2.3 Handle Missing Values and Duplicates](#23-handle-missing-values-and-duplicates)
   - [2.4 Variable Types](#24-variable-types)
   - [2.5 Check Variables Overlaps and Inconsistencies](#25-check-variables-overlaps-and-inconsistencies)
     - [2.5.1 Application Mode vs Nationality vs International Flag](#251-application-mode-vs-nationality-vs-international-flag)
     - [2.5.2 Scholarship Holder & Tuition Fees up to Date & Debtor](#252-scholarship-holder--tuition-fees-up-to-date--debtor)
     - [2.5.3 Previous Qualification & Application Mode (4, 15, 17)](#253-previous-qualification--application-mode-4-15-17)
     - [2.5.4 Age at Enrollment & Application Mode (12)](#254-age-at-enrollment--application-mode-12)
     - [2.5.5 Curricular Units (Enrolled) & Curricular Units (Approved) & Curricular Units (Evaluations)](#255-curricular-units-enrolled--curricular-units-approved--curricular-units-evaluations)
     - [2.5.6 Curricular Units (Grade) & Curricular Units (Approved)](#256-curricular-units-grade--curricular-units-approved)
     - [2.5.7 Remove Irrelevant Variables](#257-remove-irrelevant-variables)
   - [2.6 Variables Grouping](#26-variables-grouping)
   - [2.7 Recode Target Variable](#27-recode-target-variable)
3. [Conclusion](#3-conclusion)

---
## 1. Introduction

This notebook is the first part of the project focused on predicting student dropout in higher 
education. The dataset contains academic performance, socioeconomic, and demographic variables collected 
from a higher institution. This notebook covers exclusively the data understanding and preparation phase, 
including data structure inspection, handling of missing values and duplicates, variable type 
identification and grouping, and target variable recoding, ensuring the dataset is clean and 
well-structured for the subsequent EDA, feature selection and modelling stages.

---
## 2. Data Understanding & Preparation


The goal of this section is to ensure the dataset is trustworthy and ready for analysis. We will provide an overview of the data by examining its size, columns, variable types, and unique values. This process will help us identify immediate red flags, determine variables that are irrelevant to the analysis, and highlight those that require adjustments to facilitate a smoother analytical process.

In [1]:
#Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
import subprocess
import sys
import os
import kagglehub

c:\Users\mtere\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
### 2.1 Load the dataset & preview

The dataste was imported directly trough an API from kaggle. 

Then displayed the first rows to get the initial view of the data.

In [2]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])


# Download latest version
path = kagglehub.dataset_download("thedevastator/higher-education-predictors-of-student-retention")
print("Path to dataset files:", path)


# List files in the downloaded path
print(os.listdir(path))

# Read the dataset
df = pd.read_csv(path + "/dataset.csv")
display(df.head())

Path to dataset files: C:\Users\mtere\.cache\kagglehub\datasets\thedevastator\higher-education-predictors-of-student-retention\versions\2
['dataset.csv']


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


At first look, the dataset appears clean: the variable names are well-structured, and no missing data is observed in the displayed rows. However, a more thorough analysis is needed to confirm this across the entire dataset.

---

### 2.2 Inspect Data Struture

The `check()` function inspects a dataset for data quality. It returns each column’s data type, number of unique values, non-null values, missing values, and duplicates.

Addicionally, we will use the `applied data.describe().T` function to get a quick statistical summary of all numeric columns, including count, mean, standard deviation, min, max, and quartiles.

Also we will to check the unique values of each of variable to understand it better.

Together, these three steps give a clear overview of the dataset’s structure and quality.

In [3]:
def check(df):
   # Store column statistics
    list=[]

    for col in df.columns:
        columns = df.columns

        # Column data type
        dtype = df[col].dtypes

        # Non-null values
        instances = df[col].count()

        # Unique values
        unique = df[col].nunique()

        # Missing values
        sum_null = df[col].isnull().sum()

        # Duplicate values
        duplicates = df[col].duplicated().sum()
        list.append([dtype,instances,unique,sum_null,duplicates])
    data_check = pd.DataFrame(list,columns=["dtype","instances","unique","sum_null","duplicates"],index=df.columns)
    return data_check

check(df)

,dtype,instances,unique,sum_null,duplicates
Marital status,int64,4424,6,0,4418
Application mode,int64,4424,18,0,4406
Application order,int64,4424,8,0,4416
Course,int64,4424,17,0,4407
Daytime/evening attendance,int64,4424,2,0,4422
Previous qualification,int64,4424,17,0,4407
Nacionality,int64,4424,21,0,4403
Mother's qualification,int64,4424,29,0,4395
Father's qualification,int64,4424,34,0,4390
Mother's occupation,int64,4424,32,0,4392


In [4]:
# The different unique values of each variable
print(df.apply(lambda col: col.unique()))

Marital status                                                                   [1, 2, 4, 3, 5, 6]
Application mode                                  [8, 6, 1, 12, 9, 17, 15, 16, 14, 4, 13, 7, 3, ...
Application order                                                          [5, 1, 2, 4, 3, 6, 9, 0]
Course                                            [2, 11, 5, 15, 3, 17, 12, 10, 14, 16, 6, 8, 13...
Daytime/evening attendance                                                                   [1, 0]
Previous qualification                            [1, 12, 16, 14, 8, 3, 15, 2, 4, 9, 17, 11, 6, ...
Nacionality                                       [1, 15, 3, 14, 12, 18, 5, 11, 8, 17, 4, 9, 13,...
Mother's qualification                            [13, 1, 22, 23, 3, 4, 27, 2, 19, 10, 25, 7, 5,...
Father's qualification                            [10, 3, 27, 28, 1, 14, 5, 4, 24, 2, 29, 9, 7, ...
Mother's occupation                               [6, 4, 10, 8, 5, 2, 16, 1, 7, 3, 12, 9, 20, 28...


In [5]:
df.columns

Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance', 'Previous qualification', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Displaced',
       'Educational special needs', 'Debtor', 'Tuition fees up to date',
       'Gender', 'Scholarship holder', 'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd sem (grade)',
       'Curricular units 2nd sem (without evaluations)

The dataset contains **4,424 instances** and **35 variables**, including the target variable. The dataset is composed primarily of **int64** variables, with a smaller set of **float64** variables covering academic grades and macroeconomic indicators, and a single **string** variable representing the target. Overall, the dataset is well-structured and requires minimal cleaning before proceeding to the preparation and modelling stages.

---

### 2.3 Haddle Missing Values and Duplicates 


**Null or Missing Data**

There are no null values or missing values. Observing the dataset *instances* column is always equal to the number of rows (4424) and *sum_null* column shows 0 for all variables. Thus, no imputation or deletion is needed for our dataset.

**Duplicates Values**

In the table above, we observe that some variables have a lot of duplicates values but they are not row duplicates, just repeated values within columns which is normal for categorical variables.


---

### 2.4 Variable Types


**Variables Type**

Variable types can be determined based on their data type, number of unique values, and the nature of the data. **Continuous variables** are represented as float64 and capture measurements that can take any decimal value, such as curricular unit grades and macroeconomic indicators like unemployment rate, inflation rate, and GDP. **Discrete variables** are represented as int64 and capture countable numeric values, including age at enrollment and all curricular unit counts across both semesters.

**Categorical variables** are represented as int64 or string (object) with no meaningful numeric order, such as marital status, course, nationality, parental qualifications and occupations, and the target variable (Dropout, Graduate, Enrolled). **Ordinal variables** are int64 where values follow a meaningful ranked order, as seen in previous qualification and application order. Finally, **binary variables** are int64 with only two possible values (0 or 1), representing conditions such as displaced, debtor, gender, scholarship holder, and daytime/evening attendance, among others.


This variety will allow for comprehensive analysis across different data types in the EDA.


---

### 2.5 Check Variables overlaps and inconsistencies

#### 2.5.1 Application Mode vs Nationality vs International Flag

**Rule:** Students who applied via international application modes (6 = "International student (bachelor)" or 18 = "Change in institution/course (International)") should have:
- `International = 1`
- `Nacionality ≠ 1` (not Portuguese)

In [6]:
inconsistent_mask = (
    (df['Application mode'].isin([6, 18])) &
    (df['Nacionality'] == 1) &
    (df['International'] == 0)
)

inconsistent_records = df[inconsistent_mask]
print(f"Inconsistent records detected: {len(inconsistent_records)}")
print(inconsistent_records[['Application mode', 'Nacionality', 'International']])

Inconsistent records detected: 3
      Application mode  Nacionality  International
1                    6            1              0
1409                 6            1              0
3502                 6            1              0


**Finding:** 3 students have Portuguese nationality (`Nacionality = 1`) and are flagged as non-international (`International = 0`), despite applying through an international application mode (6 or 18).

This suggests these students were living abroad at enrollment time but hold Portuguese nationality, creating a 3-way inconsistency across `Application mode`, `International`, and `Nacionality`.

**Decision:** Drop these 3 rows as all three variables contradict each other and no reliable correction can be made without additional information.

In [7]:
# Drop the 3 ambiguous records
df = df[~inconsistent_mask]

print(f"Rows dropped: {inconsistent_mask.sum()}")
print(f"Dataset shape after fix: {df.shape}")

Rows dropped: 3
Dataset shape after fix: (4421, 35)


#### 2.5.2 Scholarship holder & Tuition fees up to date & Debtor

- `Debtor`: Student has **any** outstanding debt with the institution (including old/historical unpaid fees)
- `Tuition fees up to date`: Student's **current semester** fees are paid or processed
- `Scholarship holder`: Student receives a scholarship (binary — does not distinguish full vs partial)

**Expected Valid Combinations**
| Scholarship holder | Debtor | Tuition fees up to date | Interpretation | 
|---|---|---|---|
| 1 | 0 | 1 | Full scholarship, all fees covered | 
| 1 | 1 | 0 | Partial scholarship, remainder still owed | 
| 1 | 1 | 1 | Old debt exists but current tuition covered by scholarship | 
| 1 | 0 | 0 | No old debt, current tuition not yet processed | 
| 0 | 1 | 0 | No scholarship, has unpaid fees | 
| 0 | 0 | 1 | No scholarship, fees fully paid | 
| 0 | 1 | 1 | Old debt exists but current tuition is paid | 
| 0 | 0 | 0 | No old debt, current tuition not yet processed | 

**What Would Be Truly Suspicious**
| Scholarship holder | Debtor | Tuition fees up to date | Why Suspicious |
|---|---|---|---|
| 1 | 0 | 0 | Scholarship active, no debt, yet current tuition not paid — scholarship should cover it | 
| 1 | 1 | 0 | Partial scholarship but BOTH old debt AND current tuition unpaid — double financial stress unlikely for a scholar | 
| 0 | 0 | 0 | No scholarship, no registered debt yet tuition not up to date — fee simply missing from records | 

In [8]:
# Full analysis: Debtor vs Tuition fees up to date vs Scholarship holder

# Global combinations
suspicious_1 = len(df[(df['Debtor'] == 1) & (df['Tuition fees up to date'] == 1)])
suspicious_2 = len(df[(df['Debtor'] == 0) & (df['Tuition fees up to date'] == 0)])

print(f"Has debt but tuition up to date: {suspicious_1}")
print(f"No debt but tuition NOT up to date: {suspicious_2}")

# Breakdown by scholarship status
print("\n Breakdown by Scholarship Status")
for s in [0, 1]:
    label = "Scholarship" if s == 1 else "No Scholarship"
    s1 = len(df[(df['Scholarship holder'] == s) & (df['Debtor'] == 1) & (df['Tuition fees up to date'] == 1)])
    s2 = len(df[(df['Scholarship holder'] == s) & (df['Debtor'] == 0) & (df['Tuition fees up to date'] == 0)])
    print(f"[{label}] Old debt but current tuition paid: {s1}")
    print(f"[{label}] No old debt but current tuition not yet processed: {s2}")

# Truly suspicious combinations
print("\nTruly Suspicious Combinations")
sus_a = len(df[
    (df['Scholarship holder'] == 1) &
    (df['Debtor'] == 0) &
    (df['Tuition fees up to date'] == 0)
])
sus_b = len(df[
    (df['Scholarship holder'] == 1) &
    (df['Debtor'] == 1) &
    (df['Tuition fees up to date'] == 0)
])
sus_c = len(df[
    (df['Scholarship holder'] == 0) &
    (df['Debtor'] == 0) &
    (df['Tuition fees up to date'] == 0)
])

print(f"Scholarship + no debt + tuition NOT up to date: {sus_a}")
print(f"Scholarship + old debt + current tuition NOT paid: {sus_b}")
print(f"No scholarship + no debt + tuition NOT up to date: {sus_c}")

Has debt but tuition up to date: 257
No debt but tuition NOT up to date: 280

 Breakdown by Scholarship Status
[No Scholarship] Old debt but current tuition paid: 196
[No Scholarship] No old debt but current tuition not yet processed: 257
[Scholarship] Old debt but current tuition paid: 61
[Scholarship] No old debt but current tuition not yet processed: 23

Truly Suspicious Combinations
Scholarship + no debt + tuition NOT up to date: 23
Scholarship + old debt + current tuition NOT paid: 23
No scholarship + no debt + tuition NOT up to date: 257


Two levels of financial distress were identified. The most severe cases are students who carry old debt and have current tuition unpaid 23 scholarship holders whose partial scholarship is not enough to cover all their financial obligations. 

A milder level of stress is seen in 196 non-scholarship students and 61 scholarship holders who carry old debt but manage to keep current tuition paid. 

These financial distress patterns are worth monitoring as financial pressure is a likely driver of student dropout.

No inconsistencies found. The apparent contradictions are fully explained by the difference in what each variable measures: `Debtor` tracks any historical debt while `Tuition fees up to date` tracks only the current semester. No fix is applied only understading regards financial distress.

#### 2.5.3 Previous qualification & Application mode (4, 15, 17)

Certain application modes imply a specific previous qualification:
- **Mode 4** = "Holders of other higher courses" → Previous qualification should be any higher education (2, 3, 4, 5, 6, 15, 17)
- **Mode 15** = "Technological specialization diploma holders" → Previous qualification should be 14 (Technological specialization course)
- **Mode 17** = "Short cycle diploma holders" → Previous qualification should be 15 (Higher education — degree 1st cycle)


*Note on Mode 17 and Professional Higher Technical Course*

In Portugal, the Professional higher technical course (code 16) is part of the secondary school system but represents a more advanced and technical track. It is the closest equivalent to a short cycle diploma at secondary level and is therefore considered borderline valid for application mode 17.

In [9]:
# Define valid previous qualifications per application mode
rules = {
    4:  [2, 3, 4, 5, 6, 15, 17],  # any higher education
    15: [14],                       # technological specialization
    17: [15, 16]                    # short cycle / 1st cycle degree + professional technical (borderline valid in Portugal)
}

# Check each mode
for mode, valid_quals in rules.items():
    inconsistent = df[
        (df['Application mode'] == mode) &
        (~df['Previous qualification'].isin(valid_quals))
    ]
    print(f"Mode {mode} — inconsistent records: {len(inconsistent)}")
    if len(inconsistent) > 0:
        print(inconsistent['Previous qualification'].value_counts())

# Borderline valid: mode 17 with Professional higher technical course (code 16)
borderline = len(df[
    (df['Application mode'] == 17) &
    (df['Previous qualification'] == 16)
])
print(f"\nMode 17 — borderline valid (Professional higher technical course): {borderline}")

# Total to drop: previous qualification = 1 across modes 4, 15, 17
to_drop = df[
    (df['Application mode'].isin([4, 15, 17])) &
    (df['Previous qualification'] == 1)
]
print(f"\nTotal records to drop: {len(to_drop)}")

Mode 4 — inconsistent records: 1
Previous qualification
1    1
Name: count, dtype: int64
Mode 15 — inconsistent records: 13
Previous qualification
1    13
Name: count, dtype: int64
Mode 17 — inconsistent records: 2
Previous qualification
1    2
Name: count, dtype: int64

Mode 17 — borderline valid (Professional higher technical course): 33

Total records to drop: 16



| Application Mode | Inconsistent Records | Previous Qualification | Verdict |
|---|---|---|---|
| 4 | 1 | 1 — Secondary education | ❌ Drop |
| 15 | 13 | 1 — Secondary education | ❌ Drop |
| 17 | 33 | 16 — Professional higher technical course | ✅ Keep (borderline valid in Portugal) |
| 17 | 2 | 1 — Secondary education | ❌ Drop |

**Total records dropped: 16**
**Records kept (borderline valid): 33**

In [10]:
# Drop students with secondary education (code 1) applying through modes 4, 15, or 17
inconsistent_mask = (
    (df['Application mode'].isin([4, 15, 17])) &
    (df['Previous qualification'] == 1)
)

count = inconsistent_mask.sum()
df = df[~inconsistent_mask]

print(f"Rows dropped: {count}")
print(f"Dataset shape after fix: {df.shape}")

Rows dropped: 16
Dataset shape after fix: (4405, 35)


#### 2.5.4 Age at enrollment & Application mode (12)

Application mode 12 = "Over 23 years old" is a specific access route for mature students. Every student with this application mode should have Age at enrollment >= 23.

In [11]:
print(df[df['Application mode'] == 12]['Age at enrollment'].describe())
under23 = df[(df['Application mode'] == 12) & (df['Age at enrollment'] < 23)]
print(f"Inconsistent records (App mode 12 but under 23): {len(under23)}")

count    785.000000
mean      33.636943
std        8.413844
min       24.000000
25%       27.000000
50%       32.000000
75%       39.000000
max       70.000000
Name: Age at enrollment, dtype: float64
Inconsistent records (App mode 12 but under 23): 0


Since minimum age at enrollment is 24, no inconsistencies found. All students who applied through the mature student access route are correctly aged 23 or above. No fix needed.

#### 2.5.5 Curricular units (enrolled) & Curricular units (approved) & Curricular units (evaluations)

Logically, approved ≤ enrolled and evaluations ≤ enrolled. Any record where approved > enrolled or evaluations > enrolled is a data quality issue.

In [12]:
inc_approved = df[df[f'Curricular units 1st sem (approved)'] >  df[f'Curricular units 1st sem (enrolled)']]
inc_eval     = df[df[f'Curricular units 1st sem (evaluations)'] > df[f'Curricular units 1st sem (enrolled)']]
print(f"1st sem — approved > enrolled: {len(inc_approved)}")
print(f"1st sem — evaluations > enrolled: {len(inc_eval)}")

1st sem — approved > enrolled: 0
1st sem — evaluations > enrolled: 2782


The `Curricular units (evaluations)` variable counts total evaluation attempts including resits, meaning a student can have more evaluations than enrolled units if they retook exams. This explains the 2792 records in the 1st semester where evaluations exceed enrolled units, these are not inconsistencies but rather students who retook one or more exams. From a dropout prediction perspective, this is actually a valuable signal: students with a high number of evaluations relative to enrolled units are likely struggling academically, repeatedly failing and retaking exams, which may indicate a higher risk of dropout. Therefore, these records should be kept as-is since the pattern itself carries predictive information.

#### 2.5.6 Curricular units (grade) & Curricular units (approved)
A student with approved = 0 should have grade = 0. A non-zero grade with zero approved units is suspicious.

In [13]:
inc = df[(df['Curricular units 1st sem (approved)'] == 0) & 
         (df['Curricular units 1st sem (grade)'] > 0)]
print(f"1st sem — grade > 0 but approved = 0: {len(inc)}")

1st sem — grade > 0 but approved = 0: 0


No inconsistencies found. The grade and approved variables are fully consistent on the 1st semester. No fix needed.

### 2.5.7 Remove irrelevant variables

Since our goal is to predict student dropout after the first semester, all second semester variables are dropped, they would not be available at prediction time.

In [14]:
df = df.drop(columns=[
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (without evaluations)'
])

---

### 2.6 Variables Grouping

Before proceeding with EDA, variables were organized into thematic groups based on their conceptual relationship with the dropout phenomenon. This grouping serves to structure the exploratory analysis and ensure that each dimension of the student profile is considered independently.

**1. Demographic & Personal Background**
Captures intrinsic student characteristics that may influence academic persistence, such as age, gender, marital status, and origin.

**2. Socioeconomic & Family Background**
Reflects the family's educational and professional context, which is known in the literature to be a strong predictor of academic success and dropout risk.

**3. Enrollment & Application Profile**
Describes how and why the student entered the institution, including the course chosen, the time of attendance, and prior academic qualifications.

**4. Financial Status**
Captures the student's financial situation directly, including debt, scholarship support, and tuition payment status, variables strongly associated with dropout in higher education.

**5. Academic Performance**
Reflects early academic behavior and performance, which are among the most immediate indicators of dropout risk.

**6. Macroeconomic Context**
External economic conditions at the time of enrollment that may indirectly influence a student's ability to remain enrolled.

In [15]:
# Grouping variables

demographic = [
    'Marital status',
    'Gender',
    'Age at enrollment',
    'Nacionality',
    'International',
    'Displaced'
]

socioeconomic = [
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    'Educational special needs'
]

enrollment = [
    'Application mode',
    'Application order',
    'Course',
    'Daytime/evening attendance',
    'Previous qualification'
]

financial = [
    'Debtor',
    'Tuition fees up to date',
    'Scholarship holder'
]

academic_1st_sem = [
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)'
]


macroeconomic = [
    'Unemployment rate',
    'Inflation rate',
    'GDP'
]

# Identifying the groups
variable_groups = {
    'Demographic & Personal Background'   : demographic,
    'Socioeconomic & Family Background'   : socioeconomic,
    'Enrollment & Application Profile'    : enrollment,
    'Financial Status'                    : financial,
    '1st Semester Academic Performance'   : academic_1st_sem,
    'Macroeconomic Context'               : macroeconomic
}

# Verify if all variables were included
all_grouped = [var for group in variable_groups.values() for var in group]
all_features = [col for col in df.columns if col != 'Target']

not_grouped = [var for var in all_features if var not in all_grouped]
print(f"Total grouped: {len(all_grouped)} | Total features: {len(all_features)}")

Total grouped: 28 | Total features: 28


---

### 2.6 Recode Target variable

The target variable must be binarized since the current Target has three classes (Dropout, Graduate, Enrolled). Depending on the goal, one approach is to recode it as Dropout vs. Non-Dropout (Graduate + Enrolled), turning it into a binary classification problem.

In [16]:
# Recode Target variable: Dropout = 1, Non-Dropout = 0
df['Target_binary'] = (df['Target'] == 'Dropout').astype(int)

# Verify the recoding
print(df['Target_binary'].value_counts())
print(df[['Target', 'Target_binary']].drop_duplicates())

Target_binary
0    2988
1    1417
Name: count, dtype: int64
      Target  Target_binary
0    Dropout              1
3   Graduate              0
16  Enrolled              0


**Save Cleaned Dataset**

In [17]:
# Save your cleaned dataset into that folder
df.to_csv("../datasets/clean_dataset.csv", index=False)
print("Saved!")

Saved!


---
### 3. Conclusion


This notebook focused on the understanding and preparation of a dataset containing students enrolled in higher education in Portugal, with the ultimate goal of predicting dropout. The data was found to be largely clean, with no missing values or duplicates. The main effort was directed at identifying and resolving cross-variable inconsistencies across eight defined rules, resulting in 19 dropped records that presented irreconcilable contradictions between application mode, nationality, international flag, and previous qualification. All other apparent inconsistencies were resolved through a deeper understanding of the variables: notably the distinction between historical debt and current tuition status, and the Portuguese educational context that makes the professional higher technical course a borderline valid qualification for short cycle applicants.

Beyond data cleaning, the preparation phase also produced new analytical insights relevant to dropout prediction. The relationship between evaluations and enrolled units revealed a subset of students engaged in exam resits, which can be capture later on through a ratio variable that quantifies academic struggle intensity. Financial distress patterns were also identified, with 23 scholarship holders unable to cover all fees and 196 non-scholarship students carrying historical debt alongside unpaid current tuition. Both groups representing elevated dropout risk profiles. 

To keep things organized, variables were grouped into 6 thematic groups that reflect the different dimensions of a student's profile, making it easier to reason about them in the next stages. The target variable was also recoded into a simple binary outcome, so the dataset is now ready for feature selection and modelling.

